In [ ]:
# Cell 00: Setup and dependency check
# Halts if any required package is missing — mirrors 00_env_check.R

required_packages <- c("ggplot2", "RcppTOML", "scales", "patchwork")

missing <- required_packages[
  !sapply(required_packages, requireNamespace, quietly = TRUE)
]

if (length(missing) > 0) {
  stop(paste(
    "HALT: Missing packages:", paste(missing, collapse = ", "),
    "\nInstall with: install.packages(c(",
    paste0('"', missing, '"', collapse = ", "), "))"
  ))
}

suppressPackageStartupMessages({
  library(ggplot2)
  library(RcppTOML)
  library(scales)
  library(patchwork)
})

cat("Runtime check PASS\n")
cat("R version:", R.version$major, ".", R.version$minor, "\n", sep = "")
cat("ggplot2:", as.character(packageVersion("ggplot2")), "\n")
cat("RcppTOML:", as.character(packageVersion("RcppTOML")), "\n")


In [ ]:
# Cell 01: Load configuration
# All scenario parameters sourced from CFG. No hardcoded values.

TOML_PATH <- file.path("..", "config", "sira.toml")

if (!file.exists(TOML_PATH)) {
  stop(paste("HALT: sira.toml not found at", TOML_PATH,
             "\nRun from the notebooks/ directory or adjust TOML_PATH."))
}

CFG <- RcppTOML::parseTOML(TOML_PATH)

cat("Configuration loaded from:", normalizePath(TOML_PATH), "\n")
cat("Scenarios declared:", paste(CFG$scenarios$names, collapse = ", "), "\n")


In [ ]:
# Cell 02: Seed and runtime declaration
# Reproducibility is guaranteed by CFG$runtime$seed.
# Every rnorm() and rbeta() call downstream inherits this seed state.

SEED <- CFG$runtime$seed
N    <- CFG$runtime$n_simulations

set.seed(SEED)

cat("Seed:        ", SEED, "\n")
cat("Simulations: ", N, "\n")
cat("Data mode:   ", CFG$runtime$data_mode, "\n")
cat(
  "Reproducibility: all draws are deterministic from this seed.",
  "Re-run from Cell 02 to reset.\n"
)


In [ ]:
# Cell 03: Scenario parameter display table
# Renders a governance-ready parameter table from CFG.
# This is the declared-intent register — what the model is configured to do.

scenario_names <- CFG$scenarios$names

param_table <- do.call(rbind, lapply(scenario_names, function(s) {
  sc <- CFG$scenarios[[s]]
  data.frame(
    Scenario         = s,
    Distribution     = sc$distribution,
    Shape1           = ifelse(!is.null(sc$shape1),        sc$shape1,        NA),
    Shape2           = ifelse(!is.null(sc$shape2),        sc$shape2,        NA),
    Exponent         = ifelse(!is.null(sc$exponent),      sc$exponent,      NA),
    Ruin_Threshold   = sc$ruin_threshold,
    Shock_Multiplier = ifelse(!is.null(sc$shock_multiplier), sc$shock_multiplier, 1.0),
    FX_Devaluation   = ifelse(!is.null(sc$fx_devaluation),  sc$fx_devaluation,   NA),
    stringsAsFactors = FALSE
  )
}))

# Vol multiplier column — derived from distribution type and shock params
param_table$Vol_Multiplier <- ifelse(
  param_table$Distribution == "beta",
  ifelse(is.na(param_table$Shock_Multiplier), "1.00x",
         paste0(param_table$Shock_Multiplier, "x")),
  ifelse(!is.na(param_table$FX_Devaluation),
         paste0("fx_dev: ", param_table$FX_Devaluation),
         "exponent-implied")
)

print(param_table[, c(
  "Scenario", "Distribution", "Ruin_Threshold",
  "Shock_Multiplier", "Vol_Multiplier"
)])


In [ ]:
# Cell 04: Helper functions
# draw_beta(), draw_powerlaw(), apply_shock(), compute_signals()
# Mirror the logic in scripts/02_analysis.R exactly.
# No scenario values are hardcoded — all parameters passed explicitly.

draw_beta <- function(n, shape1, shape2, recovery_anchor) {
  # Beta draws conditioned on recovery_anchor
  # Bounded support enforced by rbeta — no clipping required at draw
  raw <- rbeta(n, shape1, shape2)
  # Shift toward recovery_anchor (portfolio-level conditioning)
  conditioned <- raw * recovery_anchor + (1 - recovery_anchor) * raw
  # Hard clip to (0.001, 0.995) — economically feasible fraction
  pmin(pmax(conditioned, 0.001), 0.995)
}

draw_powerlaw <- function(n, exponent) {
  # Inverse-transform sampling for Power Law heavy tail
  # P(X > x) = x^(-exponent) for x >= 1
  # Recovery fraction: clip to (0.001, 0.995)
  u <- runif(n)
  raw <- (1 - u)^(-1 / exponent)
  # Normalise to (0,1) recovery fraction
  normalised <- 1 / raw
  pmin(pmax(normalised, 0.001), 0.995)
}

apply_shock <- function(recoveries, shock_multiplier = 1.0,
                        fx_devaluation = NULL) {
  shocked <- recoveries * shock_multiplier
  if (!is.null(fx_devaluation) && !is.na(fx_devaluation)) {
    shocked <- shocked * (1 - fx_devaluation)
  }
  pmin(pmax(shocked, 0.001), 0.995)
}

compute_signals <- function(recoveries, ruin_threshold, zscore_threshold) {
  z    <- scale(recoveries)[, 1]
  ruin <- recoveries <= ruin_threshold
  sell <- ruin | (z <= -abs(zscore_threshold))
  ifelse(sell, "SELL", "HOLD")
}

cat("Helper functions loaded: draw_beta, draw_powerlaw,",
    "apply_shock, compute_signals\n")


In [ ]:
# Cell 05: Baseline scenario
# Distribution: Beta | Vol multiplier: 1.00x
# rnorm() overlay demonstrates departure from Gaussian — key governance point.
# Gaussian is structurally wrong for distressed recovery (CREDIT_RISK_LAYER.md §3.1)

sc    <- CFG$scenarios[["Baseline"]]
N     <- CFG$runtime$n_simulations
ANCHOR <- CFG$data$recovery_anchor_default

recoveries_baseline <- draw_beta(N, sc$shape1, sc$shape2, ANCHOR)
signals_baseline    <- compute_signals(
  recoveries_baseline, sc$ruin_threshold,
  CFG$signals$sell_zscore_threshold
)

# rnorm draws at equivalent mean/sd — for structural comparison only
# These are NOT used in SIRA signal logic
rnorm_baseline <- rnorm(N,
  mean = mean(recoveries_baseline),
  sd   = sd(recoveries_baseline)
)
rnorm_baseline <- pmin(pmax(rnorm_baseline, 0.001), 0.995)

df_baseline <- data.frame(
  recovery = c(recoveries_baseline, rnorm_baseline),
  source   = rep(c("Beta (SIRA)", "Normal (reference)"), each = N),
  signal   = c(signals_baseline, rep(NA, N))
)

p_baseline <- ggplot(df_baseline, aes(x = recovery, fill = source)) +
  geom_density(alpha = 0.55, colour = NA) +
  geom_vline(
    xintercept = sc$ruin_threshold,
    colour = "#E24B4A", linetype = "dashed", linewidth = 0.7
  ) +
  annotate("text",
    x = sc$ruin_threshold + 0.02, y = Inf,
    label = paste("Ruin threshold:", sc$ruin_threshold),
    hjust = 0, vjust = 1.4, size = 3,
    colour = "#E24B4A"
  ) +
  scale_fill_manual(
    values = c("Beta (SIRA)" = "#185FA5", "Normal (reference)" = "#888780")
  ) +
  scale_x_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  labs(
    title    = "Baseline — Beta recovery distribution vs Normal reference",
    subtitle = paste0(
      "shape1=", sc$shape1, "  shape2=", sc$shape2,
      "  ruin=", sc$ruin_threshold,
      "  n=", scales::comma(N),
      "  seed=", CFG$runtime$seed
    ),
    x    = "Stressed recovery",
    y    = "Density",
    fill = NULL,
    caption = "Normal reference shown for structural comparison only — not used in SIRA signal logic."
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

print(p_baseline)

cat("\nBaseline signal summary:\n")
print(table(signals_baseline))


In [ ]:
# Cell 06: Liquidity Crunch scenario
# Distribution: Beta | Vol multiplier: shock_multiplier
# Shock compresses recoveries toward lower bound.
# rnorm overlay shows Gaussian underestimates lower-tail mass.

sc <- CFG$scenarios[["Liquidity_Crunch"]]

recoveries_raw  <- draw_beta(N, sc$shape1, sc$shape2, ANCHOR)
recoveries_lc   <- apply_shock(recoveries_raw, sc$shock_multiplier)
signals_lc      <- compute_signals(
  recoveries_lc, sc$ruin_threshold,
  CFG$signals$sell_zscore_threshold
)

rnorm_lc <- rnorm(N,
  mean = mean(recoveries_lc),
  sd   = sd(recoveries_lc)
)
rnorm_lc <- pmin(pmax(rnorm_lc, 0.001), 0.995)

df_lc <- data.frame(
  recovery = c(recoveries_lc, rnorm_lc),
  source   = rep(c("Beta shocked (SIRA)", "Normal (reference)"), each = N)
)

p_lc <- ggplot(df_lc, aes(x = recovery, fill = source)) +
  geom_density(alpha = 0.55, colour = NA) +
  geom_vline(
    xintercept = sc$ruin_threshold,
    colour = "#E24B4A", linetype = "dashed", linewidth = 0.7
  ) +
  annotate("text",
    x = sc$ruin_threshold + 0.02, y = Inf,
    label = paste("Ruin:", sc$ruin_threshold),
    hjust = 0, vjust = 1.4, size = 3, colour = "#E24B4A"
  ) +
  scale_fill_manual(
    values = c(
      "Beta shocked (SIRA)" = "#0F6E56",
      "Normal (reference)"  = "#888780"
    )
  ) +
  scale_x_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  labs(
    title    = "Liquidity Crunch — Beta with shock multiplier",
    subtitle = paste0(
      "shape1=", sc$shape1, "  shape2=", sc$shape2,
      "  shock=", sc$shock_multiplier,
      "  ruin=", sc$ruin_threshold
    ),
    x = "Stressed recovery", y = "Density", fill = NULL,
    caption = paste0(
      "Shock multiplier ", sc$shock_multiplier,
      "x applied post-draw. Vol multiplier: ", sc$shock_multiplier, "x."
    )
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

print(p_lc)

cat("\nLiquidity Crunch signal summary:\n")
print(table(signals_lc))


In [ ]:
# Cell 07: Jurisdictional Freeze scenario
# Distribution: Beta | Vol multiplier: shock_multiplier
# Recovery collapses toward ruin threshold.
# Chart highlights the proximity collapse mechanism.

sc <- CFG$scenarios[["Jurisdictional_Freeze"]]

recoveries_raw <- draw_beta(N, sc$shape1, sc$shape2, ANCHOR)
recoveries_jf  <- apply_shock(recoveries_raw, sc$shock_multiplier)
signals_jf     <- compute_signals(
  recoveries_jf, sc$ruin_threshold,
  CFG$signals$sell_zscore_threshold
)

# Proximity to ruin — key distributional property for this scenario
proximity <- recoveries_jf - sc$ruin_threshold

df_jf <- data.frame(
  recovery  = recoveries_jf,
  proximity = proximity,
  signal    = signals_jf
)

p_jf_dist <- ggplot(df_jf, aes(x = recovery, fill = signal)) +
  geom_histogram(bins = 60, alpha = 0.8, colour = NA) +
  geom_vline(
    xintercept = sc$ruin_threshold,
    colour = "#E24B4A", linetype = "dashed", linewidth = 0.8
  ) +
  scale_fill_manual(values = c("SELL" = "#E24B4A", "HOLD" = "#185FA5")) +
  scale_x_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  labs(
    title    = "Jurisdictional Freeze — recovery vs ruin threshold",
    subtitle = paste0(
      "shape1=", sc$shape1, "  shape2=", sc$shape2,
      "  shock=", sc$shock_multiplier,
      "  ruin=", sc$ruin_threshold
    ),
    x = "Stressed recovery", y = "Count", fill = "Signal"
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

p_jf_prox <- ggplot(df_jf, aes(x = proximity, fill = signal)) +
  geom_density(alpha = 0.65, colour = NA) +
  geom_vline(xintercept = 0,
    colour = "#E24B4A", linetype = "dashed", linewidth = 0.7) +
  scale_fill_manual(values = c("SELL" = "#E24B4A", "HOLD" = "#185FA5")) +
  scale_x_continuous(labels = percent_format(accuracy = 1)) +
  labs(
    title    = "Proximity to ruin threshold",
    subtitle = "recovery − ruin_threshold  |  negative = ruin event",
    x = "Proximity", y = "Density", fill = "Signal"
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

print(p_jf_dist / p_jf_prox)

cat("\nJurisdictional Freeze signal summary:\n")
print(table(signals_jf))


In [ ]:
# Cell 08: Counterparty Default scenario
# Distribution: Power Law | Vol multiplier: exponent-implied
# Heavy tail — extreme low-recovery events materially more probable.
# rnorm overlay quantifies the tail mis-specification of Gaussian.

sc <- CFG$scenarios[["Counterparty_Default"]]

recoveries_cd <- draw_powerlaw(N, sc$exponent)
recoveries_cd <- apply_shock(recoveries_cd, sc$shock_multiplier)
signals_cd    <- compute_signals(
  recoveries_cd, sc$ruin_threshold,
  CFG$signals$sell_zscore_threshold
)

rnorm_cd <- rnorm(N,
  mean = mean(recoveries_cd),
  sd   = sd(recoveries_cd)
)
rnorm_cd <- pmin(pmax(rnorm_cd, 0.001), 0.995)

df_cd <- data.frame(
  recovery = c(recoveries_cd, rnorm_cd),
  source   = rep(c("Power Law (SIRA)", "Normal (reference)"), each = N)
)

# Log-scale density to surface tail behaviour
p_cd <- ggplot(df_cd, aes(x = recovery, colour = source)) +
  geom_density(linewidth = 0.9, fill = NA) +
  geom_vline(
    xintercept = sc$ruin_threshold,
    colour = "#E24B4A", linetype = "dashed", linewidth = 0.7
  ) +
  scale_colour_manual(
    values = c(
      "Power Law (SIRA)"   = "#993C1D",
      "Normal (reference)" = "#888780"
    )
  ) +
  scale_x_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  scale_y_log10() +
  labs(
    title    = "Counterparty Default — Power Law vs Normal (log-scale density)",
    subtitle = paste0(
      "exponent=", sc$exponent,
      "  shock=", sc$shock_multiplier,
      "  ruin=", sc$ruin_threshold,
      "  vol_multiplier=exponent-implied"
    ),
    x = "Stressed recovery", y = "Density (log scale)", colour = NULL,
    caption = paste0(
      "Log scale surfaces Power Law lower-tail excess over Normal. ",
      "Gaussian underestimates ruin-zone mass by construction."
    )
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

print(p_cd)

cat("\nCounterparty Default signal summary:\n")
print(table(signals_cd))


In [ ]:
# Cell 09: Hyper-Inflationary scenario
# Distribution: Power Law | Vol multiplier: fx_devaluation
# FX devaluation impairs real bond value post-draw.
# Chart shows pre- and post-devaluation distributions side by side.

sc <- CFG$scenarios[["Hyper_Inflationary"]]

recoveries_pre  <- draw_powerlaw(N, sc$exponent)
recoveries_pre  <- apply_shock(recoveries_pre, sc$shock_multiplier)
recoveries_post <- apply_shock(
  recoveries_pre,
  fx_devaluation = sc$fx_devaluation
)
signals_hi      <- compute_signals(
  recoveries_post, sc$ruin_threshold,
  CFG$signals$sell_zscore_threshold
)

rnorm_hi <- rnorm(N,
  mean = mean(recoveries_post),
  sd   = sd(recoveries_post)
)
rnorm_hi <- pmin(pmax(rnorm_hi, 0.001), 0.995)

df_hi <- data.frame(
  recovery = c(recoveries_pre, recoveries_post, rnorm_hi),
  source   = rep(c(
    "Power Law pre-FX",
    "Power Law post-FX (SIRA)",
    "Normal (reference)"
  ), each = N)
)

p_hi <- ggplot(df_hi, aes(x = recovery, fill = source)) +
  geom_density(alpha = 0.50, colour = NA) +
  geom_vline(
    xintercept = sc$ruin_threshold,
    colour = "#E24B4A", linetype = "dashed", linewidth = 0.7
  ) +
  annotate("text",
    x = sc$ruin_threshold + 0.02, y = Inf,
    label = paste("Ruin:", sc$ruin_threshold),
    hjust = 0, vjust = 1.4, size = 3, colour = "#E24B4A"
  ) +
  scale_fill_manual(
    values = c(
      "Power Law pre-FX"         = "#BA7517",
      "Power Law post-FX (SIRA)" = "#993C1D",
      "Normal (reference)"       = "#888780"
    )
  ) +
  scale_x_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  labs(
    title    = "Hyper-Inflationary — Power Law with FX devaluation impairment",
    subtitle = paste0(
      "exponent=", sc$exponent,
      "  fx_devaluation=", sc$fx_devaluation,
      "  shock=", sc$shock_multiplier,
      "  ruin=", sc$ruin_threshold
    ),
    x = "Stressed recovery", y = "Density", fill = NULL,
    caption = paste0(
      "FX devaluation=", sc$fx_devaluation,
      " applied to real bond value post Power Law draw. ",
      "Vol multiplier: fx_devaluation."
    )
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom")

print(p_hi)

cat("\nHyper-Inflationary signal summary:\n")
print(table(signals_hi))


In [ ]:
# Cell 10: Cross-scenario comparison
# Faceted density chart — all five scenarios in a single governed view.
# rnorm reference overlay on every facet quantifies Gaussian mis-specification
# per scenario. This is the committee-presentation chart.

scenarios_all <- list(
  Baseline             = list(r = recoveries_baseline, s = signals_baseline),
  Liquidity_Crunch     = list(r = recoveries_lc,       s = signals_lc),
  Jurisdictional_Freeze= list(r = recoveries_jf,       s = signals_jf),
  Counterparty_Default = list(r = recoveries_cd,       s = signals_cd),
  Hyper_Inflationary   = list(r = recoveries_post,     s = signals_hi)
)

df_all <- do.call(rbind, lapply(names(scenarios_all), function(nm) {
  sc_cfg <- CFG$scenarios[[nm]]
  r      <- scenarios_all[[nm]]$r
  s      <- scenarios_all[[nm]]$s
  nr     <- rnorm(length(r), mean = mean(r), sd = sd(r))
  nr     <- pmin(pmax(nr, 0.001), 0.995)

  rbind(
    data.frame(
      scenario     = nm,
      recovery     = r,
      source       = "SIRA",
      signal       = s,
      ruin         = sc_cfg$ruin_threshold,
      distribution = sc_cfg$distribution,
      stringsAsFactors = FALSE
    ),
    data.frame(
      scenario     = nm,
      recovery     = nr,
      source       = "Normal ref",
      signal       = NA_character_,
      ruin         = sc_cfg$ruin_threshold,
      distribution = "normal",
      stringsAsFactors = FALSE
    )
  )
}))

# Scenario display labels with distribution and vol multiplier
scenario_labels <- c(
  Baseline              = "Baseline\nBeta | 1.00x",
  Liquidity_Crunch      = "Liquidity Crunch\nBeta | shock_multiplier",
  Jurisdictional_Freeze = "Jurisdictional Freeze\nBeta | shock_multiplier",
  Counterparty_Default  = "Counterparty Default\nPower Law | exponent-implied",
  Hyper_Inflationary    = "Hyper-Inflationary\nPower Law | fx_devaluation"
)
df_all$scenario_label <- scenario_labels[df_all$scenario]
df_all$scenario_label <- factor(
  df_all$scenario_label,
  levels = scenario_labels
)

# Ruin threshold per facet
df_ruin <- unique(df_all[, c("scenario_label", "ruin")])

p_cross <- ggplot(
  df_all[df_all$source == "SIRA", ],
  aes(x = recovery, fill = signal)
) +
  geom_density(alpha = 0.70, colour = NA) +
  geom_density(
    data = df_all[df_all$source == "Normal ref", ],
    aes(x = recovery),
    inherit.aes = FALSE,
    fill = "#888780", alpha = 0.25, colour = NA
  ) +
  geom_vline(
    data = df_ruin,
    aes(xintercept = ruin),
    colour = "#E24B4A", linetype = "dashed", linewidth = 0.6
  ) +
  facet_wrap(~ scenario_label, ncol = 1, scales = "free_y") +
  scale_fill_manual(
    values = c("SELL" = "#E24B4A", "HOLD" = "#185FA5"),
    na.value = "#888780"
  ) +
  scale_x_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  labs(
    title   = "SIRA — Five stress scenarios: recovery distributions",
    subtitle = paste0(
      "Seed: ", CFG$runtime$seed,
      "  |  n=", scales::comma(N),
      "  |  Grey: Normal reference  |  Dashed: ruin threshold"
    ),
    x    = "Stressed recovery",
    y    = "Density",
    fill = "Signal",
    caption = paste0(
      "SIRA stochastic recovery engine. ",
      "Beta scenarios: Baseline, Liquidity Crunch, Jurisdictional Freeze. ",
      "Power Law scenarios: Counterparty Default, Hyper-Inflationary. ",
      "Normal reference shown for structural comparison only."
    )
  ) +
  theme_minimal(base_size = 11) +
  theme(
    legend.position  = "bottom",
    strip.text       = element_text(size = 9, face = "plain"),
    panel.spacing    = unit(1.2, "lines")
  )

print(p_cross)


In [ ]:
# Cell 11: SELL/HOLD signal summary table
# Governance-ready summary. Matches terminal emission format from
# scripts/03_visualize.R. Suitable for committee review.

summary_rows <- lapply(names(scenarios_all), function(nm) {
  sc_cfg  <- CFG$scenarios[[nm]]
  r       <- scenarios_all[[nm]]$r
  s       <- scenarios_all[[nm]]$s
  n_sell  <- sum(s == "SELL")
  n_hold  <- sum(s == "HOLD")
  n_total <- length(s)

  data.frame(
    Scenario        = nm,
    Distribution    = sc_cfg$distribution,
    Vol_Multiplier  = ifelse(
      sc_cfg$distribution == "beta",
      ifelse(!is.null(sc_cfg$shock_multiplier),
             paste0(sc_cfg$shock_multiplier, "x"), "1.00x"),
      ifelse(!is.null(sc_cfg$fx_devaluation),
             paste0("fx_dev: ", sc_cfg$fx_devaluation),
             "exponent-implied")
    ),
    Ruin_Threshold  = sc_cfg$ruin_threshold,
    Mean_Recovery   = round(mean(r), 4),
    SD_Recovery     = round(sd(r),   4),
    N_SELL          = n_sell,
    N_HOLD          = n_hold,
    SELL_pct        = paste0(round(100 * n_sell / n_total, 1), "%"),
    stringsAsFactors = FALSE
  )
})

summary_table <- do.call(rbind, summary_rows)

cat("\n")
cat(strrep("─", 80), "\n")
cat("SIRA — Five Scenario Signal Summary\n")
cat(strrep("─", 80), "\n")
print(summary_table, row.names = FALSE)
cat(strrep("─", 80), "\n")

# Worst-case scenario by SELL count
worst <- summary_table[which.max(summary_table$N_SELL), "Scenario"]
cat("Worst-case scenario:", worst, "\n")
cat(strrep("─", 80), "\n")


In [ ]:
# Cell 12: Session metadata and reproducibility declaration
# This cell produces the evidence record for the notebook run.
# Suitable for inclusion in compliance evidence packs.

cat(strrep("═", 80), "\n")
cat("SIRA NOTEBOOK — SESSION METADATA\n")
cat(strrep("═", 80), "\n")
cat("Timestamp (UTC): ", format(Sys.time(), tz = "UTC", usetz = TRUE), "\n")
cat("Seed:            ", CFG$runtime$seed, "\n")
cat("Simulations:     ", scales::comma(N), "\n")
cat("Config source:   ", normalizePath(TOML_PATH), "\n")
cat("R version:       ",
    R.version$major, ".", R.version$minor, "\n", sep = "")
cat("ggplot2:         ", as.character(packageVersion("ggplot2")), "\n")
cat("RcppTOML:        ", as.character(packageVersion("RcppTOML")), "\n")
cat("Scenarios run:   ", paste(names(scenarios_all), collapse = ", "), "\n")
cat(strrep("─", 80), "\n")
cat("Reproducibility: all stochastic draws are deterministic from the\n")
cat("declared seed (", CFG$runtime$seed, "). Re-run from Cell 02 to reset.\n",
    sep = "")
cat("This notebook does not write to output/ or modify any governed\n")
cat("artefact. It is a read-only inspection surface over the SIRA engine.\n")
cat(strrep("═", 80), "\n")
